# Trace Analysis

Workflow:
1. **Scan** — load results, find failures, spot patterns
2. **Read** — pick a trace, read the conversation step by step
3. **Note** — write down what you observe (open coding)
4. **Cluster** — after 20-30 traces, group your notes into failure categories
5. **Fix** — change prompts/tools/behavior, rerun, compare

In [ ]:
import json
from pathlib import Path

import pandas as pd

TRACES_DIR = Path("../traces")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)

## 1. Scan — find failures and patterns

In [ ]:
results = pd.read_csv(TRACES_DIR / "results.csv")
results["passed"] = results["passed"].astype(bool)

print(f"{len(results)} results: {results['passed'].sum()} passed, {(~results['passed']).sum()} failed")
results[["workspace", "category", "question", "passed", "tool_calls", "failed_assertions"]].sort_values("passed")

In [ ]:
# Pass rate by category
results.groupby("category")["passed"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "passed", "count": "total", "mean": "pass_rate"}
).sort_values("pass_rate")

In [ ]:
# Pass rate by model
results.groupby("model")["passed"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "passed", "count": "total", "mean": "pass_rate"}
).sort_values("pass_rate")

In [ ]:
# Spot the two failure modes: 0 tools = skipped tools, 20 tools = hit max_turns
failures = results[~results["passed"]]
failures[["model", "workspace", "question", "tool_calls", "failed_assertions", "error"]]

## 2. Read — pick a trace, step through the conversation

In [ ]:
def load_trace(model_slug: str, workspace: str, question_slug: str) -> dict:
    path = TRACES_DIR / model_slug / workspace / f"{question_slug}.json"
    return json.loads(path.read_text())


def list_traces(model_slug: str) -> pd.DataFrame:
    """List all available trace files for a model."""
    rows = []
    model_dir = TRACES_DIR / model_slug
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for trace_file in sorted(ws_dir.glob("*.json")):
            trace = json.loads(trace_file.read_text())
            rows.append({
                "workspace": ws_dir.name,
                "slug": trace_file.stem,
                "passed": trace["passed"],
                "tools": len(trace["tool_calls"]),
                "question": trace["question"][:60],
            })
    return pd.DataFrame(rows)


# List available models
print("Available models:")
for d in sorted(TRACES_DIR.iterdir()):
    if d.is_dir():
        print(f"  {d.name}")

In [ ]:
# Pick a model and list its traces
MODEL = "openrouter-qwen-qwen3-coder"  # <-- change this
list_traces(MODEL)

In [ ]:
def show_trace(trace: dict) -> None:
    """Print a trace as a readable conversation."""
    print(f"Question: {trace['question']}")
    print(f"Passed: {trace['passed']} | Tools: {len(trace['tool_calls'])}")
    if trace.get("error"):
        print(f"Error: {trace['error']}")
    print(f"Assertions: {trace['assertions']}")
    print("=" * 80)

    for i, tc in enumerate(trace["tool_calls"], 1):
        args = json.loads(tc["arguments"])
        print(f"\n--- Step {i}: {tc['name']} ---")
        for k, v in args.items():
            print(f"  {k}: {v}")
        if tc.get("error"):
            print(f"  ERROR: {tc['error']}")
        elif tc.get("hit") is not None:
            print(f"  hit: {tc['hit']} | result_count: {tc.get('result_count')}")

    print("\n" + "=" * 80)
    print("ANSWER:")
    print(trace.get("answer") or "(no answer — hit max_turns)")


# Load and read a trace — change the workspace and slug
trace = load_trace(MODEL, "federalist-papers", "which-papers-discuss-the-judiciary")
show_trace(trace)

## 3. Note — open coding

After reading each trace, write a short note about what you observed.
After 20-30 traces, patterns will emerge. Add notes as rows below.

In [ ]:
# Add a row each time you read a trace.
# After 20-30, look for clusters.
notes = pd.DataFrame([
    # {"model": MODEL, "workspace": "federalist-papers", "slug": "which-papers-discuss-the-judiciary", "note": "paginated read_file in chunks of 100 when whole file would fit"},
    # {"model": MODEL, "workspace": "world-factbook", "slug": "what-is-the-population-of-japan", "note": "answered from memory, 0 tool calls"},
], columns=["model", "workspace", "slug", "note"])
notes

## 4. Cluster — group notes into failure categories

Once you have enough notes, group them. Common categories:
- **skipped tools** — answered from memory
- **over-searching** — hit max_turns without concluding
- **pagination waste** — read_file in small chunks instead of whole file
- **wrong grounding** — found evidence but overrode it with training data
- **missed variant** — searched one term, didn't try synonyms

In [ ]:
# Once you've filled in notes above, count them by pattern
if len(notes) > 0:
    notes["note"].value_counts()
else:
    print("No notes yet — start reading traces!")

## 5. Compare — tool call patterns across traces

In [ ]:
def trace_summary(model_slug: str) -> pd.DataFrame:
    """Load all traces for a model and summarize tool usage."""
    rows = []
    model_dir = TRACES_DIR / model_slug
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for trace_file in sorted(ws_dir.glob("*.json")):
            trace = json.loads(trace_file.read_text())
            tool_names = [tc["name"] for tc in trace["tool_calls"]]
            abbreviations = {"list_files": "L", "search_files": "S", "read_file": "R",
                             "run_python": "P", "calculator": "C", "get_current_time": "T"}
            sequence = "→".join(abbreviations.get(t, "?") for t in tool_names)
            rows.append({
                "workspace": ws_dir.name,
                "question": trace["question"][:50],
                "passed": trace["passed"],
                "tools": len(tool_names),
                "sequence": sequence,
                "searches": tool_names.count("search_files"),
                "reads": tool_names.count("read_file"),
                "lists": tool_names.count("list_files"),
            })
    return pd.DataFrame(rows)

trace_summary(MODEL)

In [ ]:
# Tool calls CSV — if available (requires a collect_traces run with the latest schema)
tool_calls_csv = TRACES_DIR / "tool_calls.csv"
if tool_calls_csv.exists():
    tc_df = pd.read_csv(tool_calls_csv)
    print(f"{len(tc_df)} tool calls across {tc_df['trace_id'].nunique()} traces")

    # Error rate by tool
    tc_df["has_error"] = tc_df["error"].notna() & (tc_df["error"] != "")
    tc_df.groupby("tool")["has_error"].agg(["sum", "count", "mean"]).rename(
        columns={"sum": "errors", "count": "total", "mean": "error_rate"}
    ).sort_values("error_rate", ascending=False)
else:
    print("No tool_calls.csv yet — run: uv run python scripts/collect_traces.py")